# Test Set Generation


## Imports


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from ase.io import read, write
from tqdm import tqdm



## Load provenance and file map


In [2]:
# --- 1. Locate provenance + filemap (universe of available structures)
desc_dir = Path("desc")

if not desc_dir.exists():
    raise FileNotFoundError(
        "Missing ./desc directory. Run 01_descriptors_computation.ipynb first."
    )

provenance_candidates = list(desc_dir.glob("*_provenance*.parquet")) + list(desc_dir.glob("*_provenance*.csv"))
if not provenance_candidates:
    raise FileNotFoundError(
        f"No provenance table found in {desc_dir}. Run 01_descriptors_computation.ipynb to generate it."
    )
provenance_path = max(provenance_candidates, key=lambda p: p.stat().st_mtime)

filemap_candidates = list(desc_dir.glob("*_filemap*.json"))
if not filemap_candidates:
    raise FileNotFoundError(
        f"No filemap found in {desc_dir}. Run 01_descriptors_computation.ipynb to generate it."
    )
filemap_path = max(filemap_candidates, key=lambda p: p.stat().st_mtime)
print("Using:", provenance_path, filemap_path)

if provenance_path.suffix == ".parquet":
    meta = pd.read_parquet(provenance_path)
else:
    meta = pd.read_csv(provenance_path)
with filemap_path.open() as f:
    fmap = {int(k): v for k, v in json.load(f).items()}

universe = (
    meta.loc[:, ["file_id", "struct_id"]]
        .drop_duplicates()
        .sort_values(["file_id", "struct_id"])
)
universe["file_path"] = universe["file_id"].map(fmap)
print("Universe structures:", len(universe))


Using: desc/SOAP_C-O-S_20260403-101706_provenance.parquet desc/SOAP_C-O-S_20260403-101706_filemap.json
Universe structures: 1232


## Exclude already used structures


In [3]:
# --- 2. Collect all already used structures (from training manifests)
selected_root = Path("selected")
selected_dir = selected_root / "testset"
selected_dir.mkdir(parents=True, exist_ok=True)
manifests = sorted(
    m for m in selected_root.rglob("*_selected_manifest.csv")
    if m.name != "TEST_selected_manifest.csv" and "testset" not in m.parts
)
print("Found manifests:", manifests)

used_frames = []
for m in manifests:
    df = pd.read_csv(m, usecols=["file_path", "struct_id"])
    used_frames.append(df.assign(struct_id=df["struct_id"].astype(int)))

if used_frames:
    used_df = pd.concat(used_frames, ignore_index=True).drop_duplicates()
    remaining = universe.merge(used_df, on=["file_path", "struct_id"], how="left", indicator=True)
    remaining = remaining.loc[remaining["_merge"] == "left_only", universe.columns].reset_index(drop=True)
else:
    remaining = universe.reset_index(drop=True)

print("Already used structures:", 0 if not used_frames else len(used_df))
print("Remaining candidates:", len(remaining))


Found manifests: [PosixPath('selected/FPS_selected_manifest.csv')]
Already used structures: 1006
Remaining candidates: 226


## Build test set


In [4]:
# --- 4. Choose test set

rng = np.random.default_rng(12345)  # different seed from training set

# Option A: fixed number globally
n_test = min(2000, len(remaining))
if n_test == 0:
    raise ValueError("No remaining structures available to build a test set.")
sel_idx = rng.choice(remaining.index, size=n_test, replace=False)
test_df = remaining.loc[sel_idx].sort_values(["file_path", "struct_id"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["file_path"]).reset_index(drop=True)
if len(test_df) == 0:
    raise ValueError("Test set is empty after dropping missing file paths.")

test_df["n_atoms_hit"] = 0  # placeholder for schema compatibility

# --- 5. Save test manifest
test_manifest = selected_dir / "TEST_selected_manifest.csv"
test_df.to_csv(test_manifest, index=False)
print("Saved test manifest:", test_manifest, "with", len(test_df), "structures")

# load structures listed in TEST manifest
images = []
for fp, sid in tqdm(zip(test_df["file_path"], test_df["struct_id"]),
                    total=len(test_df), desc="Loading TEST structures"):
    images.append(read(fp, index=int(sid)))

# write merged trajectory and XYZ
traj_path = selected_dir / "TEST_selected.traj"
xyz_path = selected_dir / "TEST_selected.xyz"
write(traj_path, images)
write(xyz_path, images)

print(f"Wrote {len(images)} frames:")
print("  .traj ->", traj_path)
print("  .xyz  ->", xyz_path)


Saved test manifest: selected/testset/TEST_selected_manifest.csv with 226 structures


Loading TEST structures: 100%|██████████| 226/226 [01:58<00:00,  1.90it/s]

Wrote 226 frames:
  .traj -> selected/testset/TEST_selected.traj
  .xyz  -> selected/testset/TEST_selected.xyz
